In [10]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import glob

import functions.eddy_feedback as ef

# 6-hourly

In [11]:
path = '/home/links/ct715/data_storage/reanalysis/jra55_daily/processed_efp'

path_6h = os.path.join(path, 'k123_6h_ubar_epf-pr-QG_1MS_1958-2016.nc')
ds_6h = xr.open_dataset(path_6h)
ds_6h

<xarray.Dataset> Size: 5GB
Dimensions:      (time: 708, level: 37, lat: 73, lon: 144)
Coordinates:
  * lon          (lon) float64 1kB 0.0 2.5 5.0 7.5 ... 350.0 352.5 355.0 357.5
  * lat          (lat) float64 584B -90.0 -87.5 -85.0 -82.5 ... 85.0 87.5 90.0
  * level        (level) float64 296B 1e+03 975.0 950.0 925.0 ... 3.0 2.0 1.0
    year         (time) int64 6kB ...
  * time         (time) datetime64[ns] 6kB 1958-01-31 1958-02-28 ... 2016-12-31
Data variables: (12/17)
    u            (time, level, lat, lon) float32 1GB ...
    v            (time, level, lat, lon) float32 1GB ...
    t            (time, level, lat, lon) float32 1GB ...
    omega        (time, level, lat, lon) float32 1GB ...
    ubar         (time, level, lat) float32 8MB ...
    ep1_QG       (time, level, lat) float64 15MB ...
    ...           ...
    div1_QG_123  (time, level, lat) float64 15MB ...
    div2_QG_123  (time, level, lat) float64 15MB ...
    ep1_QG_gt3   (time, level, lat) float64 15MB ...
    ep2_QG_gt3   (time, level, lat) float64 15MB ...
    div1_QG_gt3  (time, level, lat) float64 15MB ...
    div2_QG_gt3  (time, level, lat) float64 15MB ...

In [12]:
path_daily = os.path.join(path, 'k123_daily_ubar_epf-pr-QG_1MS_1958-2016.nc')
ds_daily = xr.open_dataset(path_daily)
ds_daily

<xarray.Dataset> Size: 5GB
Dimensions:      (time: 708, level: 37, lat: 73, lon: 144)
Coordinates:
  * lon          (lon) float64 1kB 0.0 2.5 5.0 7.5 ... 350.0 352.5 355.0 357.5
  * lat          (lat) float64 584B -90.0 -87.5 -85.0 -82.5 ... 85.0 87.5 90.0
  * level        (level) float64 296B 1e+03 975.0 950.0 925.0 ... 3.0 2.0 1.0
  * time         (time) datetime64[ns] 6kB 1958-01-31 1958-02-28 ... 2016-12-31
Data variables: (12/17)
    u            (time, level, lat, lon) float32 1GB ...
    v            (time, level, lat, lon) float32 1GB ...
    t            (time, level, lat, lon) float32 1GB ...
    omega        (time, level, lat, lon) float32 1GB ...
    ubar         (time, level, lat) float32 8MB ...
    ep1_QG       (time, level, lat) float64 15MB ...
    ...           ...
    div1_QG_123  (time, level, lat) float64 15MB ...
    div2_QG_123  (time, level, lat) float64 15MB ...
    ep1_QG_gt3   (time, level, lat) float64 15MB ...
    ep2_QG_gt3   (time, level, lat) float64 15MB ...
    div1_QG_gt3  (time, level, lat) float64 15MB ...
    div2_QG_gt3  (time, level, lat) float64 15MB ...

In [13]:
import pandas as pd

time_slices = [(1958, 2016), (1979, 2016)]  #, (1979, 2014)]      # 
datasets = [
    (ds_6h, '6h'),
    (ds_daily, 'daily')
]
div1_methods = ['div1_QG', 'div1_QG_123', 'div1_QG_gt3']

results = []

for ds, freq in datasets:
    print(f"\n{'='*60}")
    print(f"Processing {freq} data")
    print('='*60)
    
    for div1 in div1_methods:
        print(f"\n  Method: {div1}")
        print('  ' + '-'*56)
        
        for times in time_slices:
            print(f"    Slice times: {times[0]}-{times[1]}")
            ds_times = ds.sel(time=slice(f"{times[0]}-01", f"{times[1]}-12"))
            
            # Calculate full-level EFP
            nh = ef.calculate_efp(ds_times, which_div1=div1, data_type='reanalysis', calc_south_hemis=False)
            sh = ef.calculate_efp(ds_times, which_div1=div1, data_type='reanalysis', calc_south_hemis=True)
            print(f"    Full-level: NH={nh:.4f}, SH={sh:.4f}")
            
            # Calculate 500hPa EFP
            nh_500 = ef.calculate_efp(ds_times, which_div1=div1, data_type='reanalysis', calc_south_hemis=False, slice_500hPa=True)
            sh_500 = ef.calculate_efp(ds_times, which_div1=div1, data_type='reanalysis', calc_south_hemis=True, slice_500hPa=True)
            print(f"    500hPa:     NH={nh_500:.4f}, SH={sh_500:.4f}")
            
            # Store in dictionary
            results.append({
                'time_freq': freq,
                'div1_method': div1,
                'period': f"{times[0]}-{times[1]}",
                'efp_nh': nh,
                'efp_sh': sh,
                'efp_nh_500': nh_500,
                'efp_sh_500': sh_500
            })

# Create DataFrame
df_efp = pd.DataFrame(results)
print("\n" + "="*60)
print("Final DataFrame:")
print(df_efp)


Processing 6h data

  Method: div1_QG
  --------------------------------------------------------
    Slice times: 1958-2016
    Full-level: NH=0.3532, SH=0.2740
    500hPa:     NH=0.3050, SH=0.2267
    Slice times: 1979-2016
    Full-level: NH=0.4260, SH=0.3266
    500hPa:     NH=0.3941, SH=0.2616

  Method: div1_QG_123
  --------------------------------------------------------
    Slice times: 1958-2016
    Full-level: NH=0.1400, SH=0.0188
    500hPa:     NH=0.1393, SH=0.0111
    Slice times: 1979-2016
    Full-level: NH=0.1780, SH=0.0318
    500hPa:     NH=0.1897, SH=0.0177

  Method: div1_QG_gt3
  --------------------------------------------------------
    Slice times: 1958-2016
    Full-level: NH=0.1906, SH=0.2460
    500hPa:     NH=0.1225, SH=0.1875
    Slice times: 1979-2016
    Full-level: NH=0.2163, SH=0.2914
    500hPa:     NH=0.1358, SH=0.2219

Processing daily data

  Method: div1_QG
  --------------------------------------------------------
    Slice times: 1958-2016
    

In [14]:
df_efp = df_efp[df_efp['period'] == '1958-2016'].reset_index(drop=True)
df_efp

,time_freq,div1_method,period,efp_nh,efp_sh,efp_nh_500,efp_sh_500
0,6h,div1_QG,1958-2016,0.3532,0.2740,0.3050,0.2267
1,6h,div1_QG_123,1958-2016,0.1400,0.0188,0.1393,0.0111
2,6h,div1_QG_gt3,1958-2016,0.1906,0.2460,0.1225,0.1875
3,daily,div1_QG,1958-2016,0.2885,0.1734,0.2418,0.1057
4,daily,div1_QG_123,1958-2016,0.1394,0.0179,0.1387,0.0111
5,daily,div1_QG_gt3,1958-2016,0.1227,0.1495,0.0683,0.0798


In [ ]:
# df_efp.to_csv('./data/1958-2016/jra55_efp_k123_1958-2016.csv', index=False)

### ✅ Matches other calcs

In [16]:
df2 = pd.read_csv('./data/jra55_efp_ALL_variations.csv')
df2 =df2[df2['div1_method'] == 'div1_QG']
df2 =df2[df2['period'] == '1958-2016'].reset_index(drop=True)
df2

,time_freq,div1_method,period,efp_nh,efp_sh,efp_nh_500,efp_sh_500
0,6h,div1_QG,1958-2016,0.3532,0.2740,0.3050,0.2267
1,daily,div1_QG,1958-2016,0.2885,0.1734,0.2418,0.1057


In [ ]:
# df2.to_csv('./data/1958-2016/jra55_efp_1958-2016.csv', index=False)